# kpi_eye_accuracy.ipynb — KPI ① 눈 감김 판정 정확도 (A, P0)

- 팀 사진 100장 (5명 × 뜸 · 반쯤 · 감음, 낮 · 밤 · 안경) — **시연 폰 기본 카메라로, 거치대 위치에서** 촬영
- 비교: **ML Kit 눈 뜸 확률** · **폰 EAR**(ML Kit 눈 윤곽) · **서버 EAR**(MediaPipe, 고정 / 개인 기준) · **CNN**
- 결과로 기준값 확정 → `sleep_detector_app/lib/config.dart` · `sleep_detector_api/config.py`, 서버 확인 방식(CNN / EAR) 결정 (3주차 수요일)
- **'졸았는데 못 잡은 비율'을 따로 표시** — 졸음 감지는 놓치는 쪽이 더 위험
- 🔴 측정용 사진은 CNN 학습에 쓰지 않음. 팀 사진 · CSV 는 **Google Drive 에만** (저장소에 올리지 않음)
- 🔴 GitHub 에 올리기 전: **모든 출력 지우기** (사진 이름 · 숫자가 출력에 남지 않게)

### 입력 CSV 2개 (같은 사진 100장으로)

| 파일 | 만드는 법 | 칸 |
|---|---|---|
| `eye_measure.csv` (폰) | 시연 폰에서 `flutter run -t lib/tools/eye_measure_main.dart` → 사진 선택 → `adb pull /sdcard/Android/data/com.example.sleep_detector_app/files/eye_measure.csv` | file, label, eye_open_prob, phone_ear, ms |
| `result.csv` (서버) | 사진을 `sleep_detector_api/images/` 에 넣고 `python services/face_service.py images` → `result_images/result.csv` | file, label, face_found, left_ear, right_ear, ear, eye_closed_prob, method, eye_width_px, result |

- 사진 이름 규칙: **`사람_상태_번호.jpg`** (예: `p1_open_01.jpg` · `p1_half_01.jpg` · `p1_closed_01.jpg`)
- CNN 열을 넣으려면 서버를 `config.USE_CNN = True` + `ml_models/eye_cnn.pt` 로 돌린다 (아니면 CNN 은 건너뜀)

**실행 순서:** 0 → 7 차례대로. 숫자(기준값 · 목표)는 0번 셀에서만 바꾼다.

## 0. 준비 — CSV 위치 · 지금 기준값

In [ ]:
import os, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

try:                                   # Colab 이면 드라이브에서 읽음
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/sleep_detector/kpi_eye'   # ← 두 CSV 를 넣은 드라이브 폴더
except ImportError:                    # PC Jupyter 면 이 노트북 옆 폴더
    DATA_DIR = os.environ.get('KPI_EYE_DIR', 'kpi_eye')
PHONE_CSV = os.path.join(DATA_DIR, 'eye_measure.csv')
SERVER_CSV = os.path.join(DATA_DIR, 'result.csv')
OUT_DIR = os.path.join(DATA_DIR, 'kpi_out')                      # 그림 · 요약표 저장
os.makedirs(OUT_DIR, exist_ok=True)

# 지금 기준값 (config.dart · config.py 와 같게 — 바꾸면 여기도 바꾼다)
APP_EYE_CLOSED_PROB = 0.2    # 앱: 두 눈 평균 뜸 확률 < 이 값 → 감김 (10/27 팀 결정)
APP_EAR_CLOSED = 0.2         # 앱: 폰 EAR < 이 값 → 감김 (useContourEar 켰을 때)
SERVER_EAR = 0.20            # 서버: EAR < 이 값 → 감음
CONFIRM_OPEN_MARGIN = 0.03   # 서버: EAR ≥ 기준 + 이 값 → '확실히 뜸' (앱 의심 취소)
CNN_CLOSED_PROB = 0.5        # 서버: CNN 감음 확률 ≥ 이 값 → 감음

# 개인 기준 (config 와 같게) — 사람마다 '뜸' 사진 CALIB_N 장으로 만들고, 그 사진은 평가에서 뺀다
CALIB_N = 5
APP_CALIB_RATIO, APP_CALIB_MIN, APP_CALIB_MAX = 0.5, 0.2, 0.4
SERVER_CALIB_RATIO, SERVER_CALIB_MIN, SERVER_CALIB_MAX = 0.7, 0.15, 0.25
PHONE_EAR_CALIB_RATIO, PHONE_EAR_CALIB_MIN, PHONE_EAR_CALIB_MAX = 0.7, 0.10, 0.30   # 폰 EAR 은 아직 config 에 없음 (제안값)

# 추천 기준값을 고르는 규칙: 놓친 비율이 이 값 이하인 기준 중 균형 정확도가 가장 높은 것
MAX_MISS = 0.05

## 1. 읽기 + 합치기

- 두 CSV 를 `file` 로 합친다. 한쪽에만 있는 사진은 표로 보여 준다 (갤러리에서 고를 때 빠졌거나 이름이 바뀐 것)
- 사람 = 파일 이름 첫 덩어리 (`p1_open_01.jpg` → `p1`), 종류 = `label` (closed / half / open)

In [ ]:
phone = pd.read_csv(PHONE_CSV, encoding='utf-8-sig')
server = pd.read_csv(SERVER_CSV, encoding='utf-8-sig')
print('폰', len(phone), '장 / 서버', len(server), '장')

df = phone.merge(server.drop(columns=['label']), on='file', how='outer', indicator=True)
only = df[df['_merge'] != 'both'][['file', '_merge']]
if len(only):
    print('⚠️ 한쪽에만 있는 사진', len(only), '장 (left_only = 폰만, right_only = 서버만)')
    display(only)
df = df[df['_merge'] == 'both'].drop(columns='_merge').copy()

df['person'] = df['file'].str.split('_').str[0]
df['label'] = df['label'].fillna('')
bad = df[~df['label'].isin(['open', 'half', 'closed'])]
if len(bad):
    print('⚠️ 이름에 open / half / closed 가 없는 사진 — 아래 계산에서 빠짐'); display(bad[['file']])
df = df[df['label'].isin(['open', 'half', 'closed'])].copy()

for c in ['eye_open_prob', 'phone_ear', 'ms', 'ear', 'eye_closed_prob', 'eye_width_px']:
    df[c] = pd.to_numeric(df.get(c), errors='coerce')
HAS_CNN = df['eye_closed_prob'].notna().any()
print('합친 사진', len(df), '장 / CNN 값', '있음' if HAS_CNN else '없음 (CNN 비교 건너뜀)')
display(pd.crosstab(df['person'], df['label'], margins=True))

## 2. 종류별 값 분포

- 상자 그림: 가운데 줄 = 중간값. **open 과 closed 상자가 겹치지 않을수록** 그 방법으로 잘 갈린다
- half(반쯤 감음)가 어느 쪽에 붙는지 본다 — ML Kit 뜸 확률은 half 를 open 쪽으로 보는 경우가 많음 (예비 시험)
- 값 없음 = ML Kit 이 얼굴 · 눈을 못 찾음 / 눈이 작아 폰 EAR 을 안 씀 (`minEyeWidthPx`) / 서버가 얼굴을 못 찾음

In [ ]:
ORDER = ['open', 'half', 'closed']
VALUES = [('eye_open_prob', 'ML Kit open prob', APP_EYE_CLOSED_PROB),
          ('phone_ear', 'phone EAR (ML Kit contour)', APP_EAR_CLOSED),
          ('ear', 'server EAR (MediaPipe)', SERVER_EAR)]
if HAS_CNN:
    VALUES.append(('eye_closed_prob', 'CNN closed prob', CNN_CLOSED_PROB))

rows = []
for col, name, _ in VALUES:
    for lab in ORDER:
        v = df.loc[df['label'] == lab, col]
        rows.append({'값': name, '종류': lab, '장수': len(v), '값 없음': int(v.isna().sum()),
                     '중간값': v.median(), '최소': v.min(), '최대': v.max()})
display(pd.DataFrame(rows).round(3))

fig, axes = plt.subplots(1, len(VALUES), figsize=(4.2 * len(VALUES), 3.6))
for ax, (col, name, thr) in zip(np.atleast_1d(axes), VALUES):
    data = [df.loc[df['label'] == lab, col].dropna() for lab in ORDER]
    ax.boxplot(data, labels=ORDER)
    for i, d in enumerate(data, 1):
        ax.scatter(np.full(len(d), i) + np.random.uniform(-0.12, 0.12, len(d)), d, s=10, alpha=0.6)
    ax.axhline(thr, color='red', ls='--', lw=1, label=f'now {thr}')
    ax.set_title(name, fontsize=10); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/2_distribution.png', dpi=90); plt.show()

## 3. 지금 기준값으로 판정

두 가지로 채점한다.

| 채점 | '감음'으로 볼 사진 | 왜 |
|---|---|---|
| **A. 완전히 감음** | closed | 미세수면(1.5초 연속 감김)이 잡아야 하는 것. half 는 빼고 채점 |
| **B. 반쯤 감음 포함** | closed + half | 폰 EAR 을 넣는 이유(반쯤 감긴 졸음). `useContourEar` 를 켤지 정할 때 |

| 칸 | 뜻 |
|---|---|
| 감음 재현율 | 감은 사진을 감음으로 잡은 비율 |
| **놓친 비율** | 감았는데 못 잡은 비율 (= 1 − 감음 재현율, 값 없음 포함) — **가장 중요** |
| 오탐률 | 뜬 사진을 감음으로 본 비율 (괜히 경고) |
| 균형 정확도 | (감음 재현율 + 뜸 재현율) ÷ 2 — 장수가 달라도 공평하게 |
| 값 없음 | 판정을 못 한 사진 수 (앱에서는 '모름' = 감김으로 안 셈 → 감은 사진이면 놓친 것으로 계산) |

In [ ]:
def methods(d, app_prob=APP_EYE_CLOSED_PROB, app_ear=APP_EAR_CLOSED, server_ear=SERVER_EAR, cnn=CNN_CLOSED_PROB):
    # 각 방법의 '감음' 판정 (True / False / NaN = 값 없음)
    def lt(s, t): return s.lt(t).where(s.notna())
    def ge(s, t): return s.ge(t).where(s.notna())
    prob = lt(d['eye_open_prob'], app_prob)
    pear = lt(d['phone_ear'], app_ear)
    # 앱 useContourEar: 뜸 확률이 있으면 '확률 < 기준 또는 EAR < 기준', 확률이 없으면 모름
    both = (prob.fillna(False) | pear.fillna(False)).where(d['eye_open_prob'].notna())
    out = {'ML Kit 뜸 확률': prob, '폰 EAR': pear, 'ML Kit + 폰 EAR': both, '서버 EAR': lt(d['ear'], server_ear)}
    if HAS_CNN:
        out['CNN'] = ge(d['eye_closed_prob'], cnn)
    return out

def score(pred, truth):
    # pred: 감음 판정(NaN = 값 없음), truth: 정말 감음인가 (bool)
    p = pred.fillna(False).astype(bool)          # 값 없음 = 감음으로 안 잡음
    pos, neg = truth, ~truth
    rec = (p & pos).sum() / max(pos.sum(), 1)
    fa = (p & neg).sum() / max(neg.sum(), 1)
    return {'장수': int(len(truth)), '감음 재현율': rec, '놓친 비율': 1 - rec, '오탐률': fa,
            '균형 정확도': (rec + 1 - fa) / 2, '값 없음': int(pred.isna().sum())}

SCORINGS = {'A. 완전히 감음': (df['label'] != 'half', df['label'] == 'closed'),
            'B. 반쯤 감음 포함': (pd.Series(True, index=df.index), df['label'] != 'open')}

def table(d, scoring_name, **thr):
    keep, truth = SCORINGS[scoring_name][0].loc[d.index], SCORINGS[scoring_name][1].loc[d.index]
    d, truth = d[keep], truth[keep]
    return pd.DataFrame({m: score(p, truth) for m, p in methods(d, **thr).items()}).T

for name in SCORINGS:
    print(f'■ {name} (지금 기준값)')
    display(table(df, name).round(3))

## 4. 기준값 바꿔 보기 + 추천 기준값

- 각 방법의 기준값을 조금씩 바꾸며 **놓친 비율(빨강)** · **오탐률(파랑)** 을 그린다. 세로 점선 = 지금 기준, 초록 = 추천
- 추천 = 놓친 비율 ≤ `MAX_MISS`(5%) 인 기준 중 균형 정확도가 가장 높은 것 (없으면 놓친 비율이 가장 낮은 것)
  - 점수가 같은 기준이 여러 개면 **지금 기준에 가장 가까운 값**
- ⚠️ 사진 100장으로 고른 값이라 **출발점**이다. 시연 폰 실제 운행(카메라 영상)으로 한 번 더 확인

In [ ]:
SWEEPS = [('ML Kit 뜸 확률', 'eye_open_prob', 'lt', np.round(np.arange(0.05, 0.96, 0.05), 2), APP_EYE_CLOSED_PROB),
          ('폰 EAR', 'phone_ear', 'lt', np.round(np.arange(0.05, 0.41, 0.01), 2), APP_EAR_CLOSED),
          ('서버 EAR', 'ear', 'lt', np.round(np.arange(0.10, 0.31, 0.01), 2), SERVER_EAR)]
if HAS_CNN:
    SWEEPS.append(('CNN', 'eye_closed_prob', 'ge', np.round(np.arange(0.1, 0.91, 0.05), 2), CNN_CLOSED_PROB))

def sweep(col, op, grid, scoring_name):
    keep, truth = SCORINGS[scoring_name]
    s = df.loc[keep, col]
    out = []
    for t in grid:
        pred = (s.lt(t) if op == 'lt' else s.ge(t)).where(s.notna())
        out.append({'기준': t, **score(pred, truth[keep])})
    return pd.DataFrame(out)

def pick(sw, now):
    # 동점이면 지금 기준에 가장 가까운 값 (근거 없이 기준을 크게 옮기지 않게)
    sw = sw.assign(거리=(sw['기준'] - now).abs())
    ok = sw[sw['놓친 비율'] <= MAX_MISS]
    return (ok.sort_values(['균형 정확도', '오탐률', '거리'], ascending=[False, True, True]) if len(ok)
            else sw.sort_values(['놓친 비율', '오탐률', '거리'])).iloc[0]

RECOMMEND = []
for scoring_name in SCORINGS:
    fig, axes = plt.subplots(1, len(SWEEPS), figsize=(4.2 * len(SWEEPS), 3.3))
    for ax, (name, col, op, grid, now) in zip(np.atleast_1d(axes), SWEEPS):
        if df[col].notna().sum() == 0:
            ax.set_title(f'{col}: no data'); continue
        sw = sweep(col, op, grid, scoring_name)
        best = pick(sw, now)
        RECOMMEND.append({'채점': scoring_name, '방법': name, '지금 기준': now, '추천 기준': best['기준'],
                          '놓친 비율': best['놓친 비율'], '오탐률': best['오탐률'], '균형 정확도': best['균형 정확도']})
        ax.plot(sw['기준'], sw['놓친 비율'], 'r-', label='miss')
        ax.plot(sw['기준'], sw['오탐률'], 'b-', label='false alarm')
        ax.axvline(now, color='gray', ls='--', lw=1); ax.axvline(best['기준'], color='green', lw=1.5)
        ax.set_title(col, fontsize=10); ax.set_ylim(-0.02, 1.02); ax.legend(fontsize=8)
    fig.suptitle('A: closed only' if scoring_name.startswith('A') else 'B: closed + half', fontsize=11)
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/4_sweep_{scoring_name[0]}.png', dpi=90); plt.show()

RECOMMEND = pd.DataFrame(RECOMMEND)
display(RECOMMEND.round(3))

## 5. 개인 기준 (출발 때 '정면 5초')

- 사람마다 **뜸 사진 `CALIB_N`(5)장**으로 기준을 만들고 (앱 `calibrate` · 서버 `/drive/calibrate` 와 같은 계산), **그 5장은 평가에서 뺀다** (정답을 미리 본 시험 방지)
  - 앱 뜸 확률: 중간값 × 0.5 → 0.2 ~ 0.4 / 서버 EAR: 중간값 × 0.7 → 0.15 ~ 0.25 / 폰 EAR(제안): 중간값 × 0.7 → 0.10 ~ 0.30
- 고정 기준과 **같은 사진**(5장을 뺀 나머지)으로 비교 → 발표의 '눈이 작은 사람' 답변 근거

In [ ]:
def personal(d):
    calib_idx, th = [], {}
    for person, g in d.groupby('person'):
        o = g[g['label'] == 'open'].sort_values('file').head(CALIB_N)
        calib_idx += list(o.index)
        def med(col, ratio, lo, hi, default):
            v = o[col].dropna()
            return float(np.clip(v.median() * ratio, lo, hi)) if len(v) >= 3 else default   # 3장 미만이면 기본값 (CALIB_MIN_FACES)
        th[person] = {'app_prob': med('eye_open_prob', APP_CALIB_RATIO, APP_CALIB_MIN, APP_CALIB_MAX, APP_EYE_CLOSED_PROB),
                      'phone_ear': med('phone_ear', PHONE_EAR_CALIB_RATIO, PHONE_EAR_CALIB_MIN, PHONE_EAR_CALIB_MAX, APP_EAR_CLOSED),
                      'server_ear': med('ear', SERVER_CALIB_RATIO, SERVER_CALIB_MIN, SERVER_CALIB_MAX, SERVER_EAR)}
    return pd.DataFrame(th).T, calib_idx

PERSONAL, calib_idx = personal(df)
print('사람별 개인 기준'); display(PERSONAL.round(3))
ev = df.drop(index=calib_idx)
print('평가 사진', len(ev), '장 (개인 기준용', len(calib_idx), '장 제외)')

def personal_pred(d):
    t = PERSONAL.loc[d['person']].set_index(d.index)
    lt = lambda s, x: s.lt(x).where(s.notna())
    prob, pear = lt(d['eye_open_prob'], t['app_prob']), lt(d['phone_ear'], t['phone_ear'])
    return {'ML Kit 뜸 확률 (개인)': prob, '폰 EAR (개인)': pear,
            'ML Kit + 폰 EAR (개인)': (prob.fillna(False) | pear.fillna(False)).where(d['eye_open_prob'].notna()),
            '서버 EAR (개인)': lt(d['ear'], t['server_ear'])}

for name, (keep, truth) in SCORINGS.items():
    d = ev[keep.loc[ev.index]]; tr = truth.loc[d.index]
    rows = {m: score(p, tr) for m, p in methods(d).items()}
    rows.update({m: score(p, tr) for m, p in personal_pred(d).items()})
    print(f'■ {name} — 고정 vs 개인 (같은 평가 사진)')
    display(pd.DataFrame(rows).T.round(3))

## 6. 사람별 · 눈 크기 · 폰 EAR vs 서버 EAR

- 사람별 균형 정확도 (채점 A, 지금 기준) — 특정 사람(안경 · 눈이 작은 사람)에서만 떨어지는지
- 눈 폭(px): 작으면 EAR 이 흔들린다. 폰 EAR 이 빈칸인 사진의 눈 폭을 본다 (`minEyeWidthPx` 20)
- 폰 EAR 과 서버 EAR 은 같은 원리지만 점 위치가 달라 **숫자 크기가 다를 수 있다** → 기준값을 따로 정하는 이유

In [ ]:
keep, truth = SCORINGS['A. 완전히 감음']
dA = df[keep]
per = {}
for person, g in dA.groupby('person'):
    per[person] = {m: score(p, truth.loc[g.index])['균형 정확도'] for m, p in methods(g).items()}
print('사람별 균형 정확도 (채점 A, 지금 기준)'); display(pd.DataFrame(per).T.round(3))

print('눈 폭(px, 서버 MediaPipe) — 폰 EAR 있음 / 없음')
display(df.groupby(df['phone_ear'].isna().map({False: '폰 EAR 있음', True: '폰 EAR 없음'}))['eye_width_px'].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for lab, c in zip(ORDER, ['tab:green', 'tab:orange', 'tab:red']):
    g = df[df['label'] == lab]
    axes[0].scatter(g['ear'], g['phone_ear'], s=14, c=c, label=lab)
    axes[1].scatter(g['eye_width_px'], g['phone_ear'], s=14, c=c, label=lab)
axes[0].set_xlabel('server EAR'); axes[0].set_ylabel('phone EAR'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('eye width px (server)'); axes[1].set_ylabel('phone EAR')
both = df[['ear', 'phone_ear']].dropna()
if len(both) > 2:
    axes[0].set_title(f'corr = {both.corr().iloc[0, 1]:.2f}', fontsize=10)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/6_phone_vs_server.png', dpi=90); plt.show()
print('ML Kit 처리 시간(ms, 사진 1장 — 원본 크기라 운행 중 카메라 영상보다 느림):', df['ms'].describe().round(0).to_dict())

## 7. 결과 정리 — config 에 넣을 값 · 발표 표

- 아래 표를 `OUT_DIR/kpi_eye_summary.csv` 로 저장한다 (드라이브)
- 결정할 것
  1. 앱 뜸 확률 기준 (`config.dart eyeClosedProb`) · 폰 EAR 기준 (`earClosed`)
  2. **`useContourEar` 켜기** — 채점 B 에서 'ML Kit + 폰 EAR' 의 놓친 비율이 'ML Kit 뜸 확률' 보다 확실히 낮고, 채점 A 오탐률이 크게 늘지 않으면 켠다
  3. 서버 EAR 기준 (`config.py EAR_THRESHOLD`)
  4. **CNN 채택** (3주차 수요일) — 채점 A 에서 CNN 균형 정확도 ≥ 서버 EAR 이면 `USE_CNN = True` (기획안 4-4)

In [ ]:
summary = pd.concat([table(df, n).assign(채점=n) for n in SCORINGS]).reset_index(names='방법')
summary = summary[['채점', '방법', '장수', '놓친 비율', '오탐률', '균형 정확도', '값 없음']]
summary.to_csv(f'{OUT_DIR}/kpi_eye_summary.csv', index=False, encoding='utf-8-sig')
RECOMMEND.to_csv(f'{OUT_DIR}/kpi_eye_recommend.csv', index=False, encoding='utf-8-sig')
display(summary.round(3))

A = RECOMMEND[RECOMMEND['채점'].str.startswith('A')].set_index('방법')['추천 기준']
B = RECOMMEND[RECOMMEND['채점'].str.startswith('B')].set_index('방법')['추천 기준']
print('── config 에 넣을 값 (추천, 사람이 보고 정함) ──')
print(f"config.dart  eyeClosedProb = {A.get('ML Kit 뜸 확률')}   (지금 {APP_EYE_CLOSED_PROB})")
print(f"config.dart  earClosed     = {B.get('폰 EAR')}   (지금 {APP_EAR_CLOSED}, 반쯤 감음 포함 채점으로)")
print(f"config.py    EAR_THRESHOLD = {A.get('서버 EAR')}   (지금 {SERVER_EAR})")
if HAS_CNN:
    s = summary[summary['채점'].str.startswith('A')].set_index('방법')['균형 정확도']
    print(f"CNN 채택: CNN {s['CNN']:.3f} vs 서버 EAR {s['서버 EAR']:.3f} →", 'USE_CNN = True' if s['CNN'] >= s['서버 EAR'] else 'EAR 유지')
print('저장:', OUT_DIR)

### 결과 기록 (측정 후 채우기)

| 항목 | 값 |
|---|---|
| 촬영일 · 폰 · 장소 / 조명 | |
| 사진 수 (사람 × open / half / closed) | |
| 채점 A 놓친 비율 — ML Kit / 서버 EAR / CNN | |
| 채점 B 놓친 비율 — ML Kit / ML Kit + 폰 EAR | |
| 확정 기준값 — eyeClosedProb / earClosed / EAR_THRESHOLD | |
| useContourEar · USE_CNN 결정 | |
| 사진 1장 처리 ms (원본) | |

- 이 결과는 **사진(원본 크기)** 기준이다. 운행 중에는 낮은 해상도의 카메라 영상을 보므로, 확정 전에 시연 폰 운행 화면에서 한 번 더 확인한다